# Descrição

Este notebook integra os resultados de expressão diferencial (DEG) com os módulos de coexpressão obtidos por WGCNA para gerar a lista final de genes a ser usada nas etapas externas de rede, especialmente STRING e Cytoscape.

A estratégia adotada é exploratória e a filtragem fina da rede será realizada posteriormente no Cytoscape. A seleção final não é baseada apenas na contagem bruta de DEGs por módulo; ela combina evidência de expressão diferencial, pertencimento modular e suporte biológico/estatístico do módulo.

A lógica do notebook é:

1. Carregar as tabelas anotadas geradas pelo notebook `004_gene_annotation.ipynb`:

   * deg_all_contrasts_annotated.csv;
   * wgcna_gene_modules_annotated.csv.

2. Filtrar DEGs prioritários usando:

   * padj < 0.10;
   * |log2FoldChange| >= 1;
   * Contrastes biologicamente relevantes para a hipótese do projeto:

     * MA1_vs_CTR;
     * MA100_vs_CTR;
     * MC1_vs_CTR;
     * MD1_vs_CTR;
     * MA100_vs_MA1;
     * MC100_vs_MC1.

3. Integrar os DEGs prioritários aos módulos WGCNA por gene_id.

4. Avaliar os módulos por três tipos de suporte:

   * Associação módulo-traço, usando as correlações entre eigengenes e variáveis experimentais;
   * Fração de genes do módulo que também são DEGs prioritários.

5. Selecionar módulos de forma permissiva, mantendo módulos com número mínimo de DEGs prioritários e ao menos um dos seguintes critérios:

   * Suporte por associação módulo-traço;
   * Suporte por fração de DEGs prioritários no módulo.

6. Gerar a lista final DEG + WGCNA, mantendo apenas DEGs prioritários pertencentes aos módulos selecionados.

7. Exportar os arquivos finais:

   * deg_wgcna_selected_for_string.csv: tabela gene-contraste com os genes selecionados;
   * selected_modules_summary.csv: resumo dos módulos selecionados;
   * string_input_selected_genes.txt: lista de símbolos gênicos para consulta no STRING;
   * cytoscape_node_table.csv: tabela de atributos dos nós para importação no Cytoscape.

A definição final de hubs, sub-redes e genes prioritários será realizada no Cytoscape, usando métricas topológicas, atributos de DEG, associação modular e resultados de enriquecimento funcional.


## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
ANNOTATION_DIR = PROCESSED_DIR / "annotation"
WGCNA_DIR = PROCESSED_DIR / "wgcna"
NETWORK_EXPORT_DIR = PROCESSED_DIR / "network_export"

NETWORK_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Entradas principais
DEG_ALL_ANNOTATED_PATH = ANNOTATION_DIR / "deg_all_contrasts_annotated.csv"
WGCNA_MODULES_ANNOTATED_PATH = ANNOTATION_DIR / "wgcna_gene_modules_annotated.csv"
MODULE_SUMMARY_PATH = WGCNA_DIR / "wgcna_module_summary.csv"

# Saídas finais
DEG_WGCNA_SELECTED_PATH = NETWORK_EXPORT_DIR / "deg_wgcna_selected_for_string.csv"
STRING_INPUT_SELECTED_PATH = NETWORK_EXPORT_DIR / "string_input_selected_genes.txt"
CYTOSCAPE_NODE_TABLE_PATH = NETWORK_EXPORT_DIR / "cytoscape_node_table.csv"
SELECTED_MODULES_SUMMARY_PATH = NETWORK_EXPORT_DIR / "selected_modules_summary.csv"

# Critérios DEG
ALPHA = 0.10
ABS_LOG2FC_MIN = 1.0

# Contrastes priorizados para responder à hipótese do projeto
PRIORITY_CONTRASTS = [
    "MA1_vs_CTR",
    "MA100_vs_CTR",
    "MC1_vs_CTR",
    "MD1_vs_CTR",
    "MA100_vs_MA1",
    "MC100_vs_MC1",
]

# Critérios WGCNA
# Mantidos permissivos porque a filtragem fina será feita no Cytoscape.
MODULE_TRAITS_TO_USE = [
    "particle_size_um",
    "concentration_gL",
    "is_100nm",
]

MODULE_TRAIT_PADJ_ALPHA = 0.10
MODULE_TRAIT_PVAL_ALPHA = 0.05

# Evita selecionar módulos sem sinal mínimo de DEG.
MIN_N_DEG_PRIORITY = 5

# Permissivo: permite módulos grandes se houver fração mínima de DEGs.
MIN_FRACTION_DEG_PRIORITY = 0.01

print("Diretório de exportação:", NETWORK_EXPORT_DIR)

Diretório de exportação: ../../data/interim/network_export


## Carregamento dos Dados

In [3]:
deg_df = pd.read_csv(DEG_ALL_ANNOTATED_PATH)
wgcna_df = pd.read_csv(WGCNA_MODULES_ANNOTATED_PATH)

print("DEG anotado:", deg_df.shape)
print("WGCNA anotado:", wgcna_df.shape)

display(deg_df.head())
display(wgcna_df.head())

DEG anotado: (121760, 12)
WGCNA anotado: (12176, 6)


,gene_id,gene_symbol,gene_name,contrast,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,significant,direction
0,ENSG00000185112,FAM43A,family with sequence similarity 43 member A,MA100_vs_CTR,1161.677550,1.476934,0.188463,7.836729,4.624347e-15,4.102258e-11,True,up_in_MA100
1,ENSG00000135046,ANXA1,annexin A1,MA100_vs_CTR,4815.356289,-0.970155,0.136694,-7.097251,1.272628e-12,5.142116e-09,True,up_in_CTR
2,ENSG00000040275,SPDL1,spindle apparatus coiled-coil protein 1,MA100_vs_CTR,2053.584631,1.214709,0.172202,7.053960,1.738964e-12,5.142116e-09,True,up_in_MA100
3,ENSG00000214357,NEURL1B,neuralized E3 ubiquitin protein ligase 1B,MA100_vs_CTR,1332.745788,1.313536,0.196719,6.677235,2.434923e-11,5.400051e-08,True,up_in_MA100
4,ENSG00000147133,TAF1,TATA-box binding protein associated factor 1,MA100_vs_CTR,1810.476148,1.281242,0.192941,6.640600,3.124082e-11,5.542746e-08,True,up_in_MA100


,gene_id,gene_symbol,gene_name,dynamic_module,module,module_label
0,ENSG00000000003,TSPAN6,tetraspanin 6,dimgrey,dimgrey,10
1,ENSG00000000419,DPM1,dolichyl-phosphate mannosyltransferase subunit...,mistyrose,mistyrose,17
2,ENSG00000000457,SCYL3,SCY1 like pseudokinase 3,dimgrey,dimgrey,10
3,ENSG00000000460,FIRRM,FIGNL1 interacting regulator of recombination ...,dimgrey,dimgrey,10
4,ENSG00000001036,FUCA2,alpha-L-fucosidase 2,indianred,indianred,13


In [4]:
# Garante uma única linha de módulo por gene
wgcna_df = (
    wgcna_df
    .sort_values(["gene_id"])
    .drop_duplicates(subset="gene_id", keep="first")
    .reset_index(drop=True)
)

print("Genes únicos em DEG:", deg_df["gene_id"].nunique())
print("Genes únicos em WGCNA:", wgcna_df["gene_id"].nunique())
print("Contrastes em DEG:", deg_df["contrast"].nunique())

Genes únicos em DEG: 12176
Genes únicos em WGCNA: 12176
Contrastes em DEG: 10


## Integração DEG x WGCNA

In [5]:
# Verifica se as colunas necessárias estão presentes antes de prosseguir
required_deg_cols = {
    "gene_id",
    "contrast",
    "log2FoldChange",
    "padj",
    "gene_symbol",
}

required_wgcna_cols = {
    "gene_id",
    "module",
}

missing_deg = required_deg_cols - set(deg_df.columns)
missing_wgcna = required_wgcna_cols - set(wgcna_df.columns)

if missing_deg:
    raise ValueError(f"Colunas ausentes em deg_df: {missing_deg}")

if missing_wgcna:
    raise ValueError(f"Colunas ausentes em wgcna_df: {missing_wgcna}")

# Evita colisão com anotações vindas do WGCNA
wgcna_cols_to_add = [
    col for col in wgcna_df.columns
    if col not in {"gene_symbol", "gene_name"}
]

# Faz o merge mantendo todas as linhas de deg_df e adicionando as colunas de wgcna_df
# quando houver correspondência
integrated_df = deg_df.merge(
    wgcna_df[wgcna_cols_to_add],
    on="gene_id",
    how="left",
    validate="many_to_one",
)

integrated_df["module"] = integrated_df["module"].astype("string")

integrated_df["log2FoldChange"] = pd.to_numeric(
    integrated_df["log2FoldChange"],
    errors="coerce",
)

integrated_df["padj"] = pd.to_numeric(
    integrated_df["padj"],
    errors="coerce",
)

integrated_df["abs_log2FoldChange"] = integrated_df["log2FoldChange"].abs()

# Identifica quais linhas correspondem aos contrastes prioritários
integrated_df["is_priority_contrast"] = integrated_df["contrast"].isin(PRIORITY_CONTRASTS)

# Define os DEGs prioritários com base nos critérios estabelecidos
integrated_df["is_deg_priority"] = (
    integrated_df["is_priority_contrast"]
    & integrated_df["padj"].notna()
    & (integrated_df["padj"] < ALPHA)
    & (integrated_df["abs_log2FoldChange"] >= ABS_LOG2FC_MIN)
    & integrated_df["module"].notna()
)

# Classifica a regulação com base no log2FoldChange para os DEGs prioritários
integrated_df["regulation"] = np.select(
    [
        integrated_df["is_deg_priority"] & (integrated_df["log2FoldChange"] > 0),
        integrated_df["is_deg_priority"] & (integrated_df["log2FoldChange"] < 0),
    ],
    [
        "up_in_tested",
        "down_in_tested",
    ],
    default="not_selected",
)

# Filtra apenas os DEGs prioritários para análise posterior
priority_deg_df = integrated_df.loc[
    integrated_df["is_deg_priority"]
].copy()

print("Linhas gene-contraste:", priority_deg_df.shape[0])
print("Genes únicos:", priority_deg_df["gene_id"].nunique())
print("Contrastes representados:", priority_deg_df["contrast"].nunique())
print("Módulos representados:", priority_deg_df["module"].nunique())

display(
    priority_deg_df
    .groupby("contrast")
    .agg(
        n_rows=("gene_id", "size"),
        n_unique_genes=("gene_id", "nunique"),
        n_modules=("module", "nunique"),
    )
    .reset_index()
    .sort_values("n_unique_genes", ascending=False)
)

Linhas gene-contraste: 853
Genes únicos: 610
Contrastes representados: 6
Módulos representados: 13


,contrast,n_rows,n_unique_genes,n_modules
3,MC100_vs_MC1,419,419,10
0,MA100_vs_CTR,173,173,9
4,MC1_vs_CTR,149,149,6
5,MD1_vs_CTR,96,96,6
1,MA100_vs_MA1,12,12,3
2,MA1_vs_CTR,4,4,3


## Ranqueamento dos Módulos WGCNA

In [6]:
# Resumo por módulo:
# quantos DEGs prioritários, quantos genes totais, fração de DEGs, contrastes representados, etc
module_rows = []

for module_name, module_gene_df in wgcna_df.groupby("module", dropna=False):
    if pd.isna(module_name):
        continue

    module_name = str(module_name)

    module_genes = set(
        module_gene_df["gene_id"]
        .dropna()
        .astype(str)
    )

    module_priority_df = priority_deg_df.loc[
        priority_deg_df["module"].astype(str) == module_name
    ]

    priority_genes_in_module = set(
        module_priority_df["gene_id"]
        .dropna()
        .astype(str)
    )

    contrasts_present = (
        module_priority_df["contrast"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    n_genes_module = len(module_genes)
    n_deg_priority = len(priority_genes_in_module)

    module_rows.append({
        "module": module_name,
        "n_unique_genes_in_module": n_genes_module,
        "n_deg_priority": n_deg_priority,
        "fraction_deg_priority": (
            n_deg_priority / n_genes_module
            if n_genes_module > 0
            else np.nan
        ),
        "n_priority_gene_contrast_hits": module_priority_df.shape[0],
        "n_priority_contrasts_present": len(contrasts_present),
        "priority_contrasts_present": ";".join(contrasts_present),
        "n_up_priority_rows": int((module_priority_df["regulation"] == "up_in_tested").sum()),
        "n_down_priority_rows": int((module_priority_df["regulation"] == "down_in_tested").sum()),
    })

module_ranking_df = pd.DataFrame(module_rows)

module_ranking_df["fraction_support"] = (
    module_ranking_df["fraction_deg_priority"] >= MIN_FRACTION_DEG_PRIORITY
)

print("Resumo de DEGs prioritários por módulo:")
display(
    module_ranking_df
    .sort_values(
        ["n_deg_priority", "fraction_deg_priority"],
        ascending=[False, False],
    )
    .head(20)
)

Resumo de DEGs prioritários por módulo:


,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows,fraction_support
10,dimgrey,3883,302,0.077775,443,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,204,239,True
7,darkgrey,3266,203,0.062156,263,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,115,148,True
12,gainsboro,1124,46,0.040925,76,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,51,25,True
17,mistyrose,2130,38,0.017840,44,3,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,34,10,True
13,indianred,216,6,0.027778,6,1,MC100_vs_MC1,6,0,True
28,white,317,6,0.018927,7,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,3,4,True
3,brown,158,2,0.012658,2,2,MA100_vs_CTR;MC100_vs_MC1,2,0,True
16,maroon,347,2,0.005764,7,5,MA100_vs_CTR;MA1_vs_CTR;MC100_vs_MC1;MC1_vs_CT...,6,1,False
8,darkorange,13,1,0.076923,1,1,MA100_vs_CTR,1,0,True
1,bisque,14,1,0.071429,1,1,MD1_vs_CTR,1,0,True


## Integração com Associação Módulo-Traço

In [7]:
# Se o resumo módulo-trait estiver disponível, faz o merge para enriquecer a seleção
if MODULE_SUMMARY_PATH.exists():
    module_trait_df = pd.read_csv(MODULE_SUMMARY_PATH)

    module_trait_df["module"] = (
        module_trait_df["module"]
        .astype(str)
        .str.replace(r"^ME_", "", regex=True)
    )

    module_ranking_df = module_ranking_df.merge(
        module_trait_df,
        on="module",
        how="left",
        validate="one_to_one",
    )

    print("Resumo módulo-trait carregado:", MODULE_SUMMARY_PATH)
else:
    print("Aviso: wgcna_module_summary.csv não encontrado. A seleção usará apenas suporte DEG/módulo.")

Resumo módulo-trait carregado: ../../data/interim/wgcna/wgcna_module_summary.csv


## Seleção de Módulos

In [8]:
# A seleção final é centrada em DEGs prioritários e usa o WGCNA como camada de contexto modular.
# Os módulos foram mantidos quando concentraram uma fração mínima de DEGs prioritários e/ou apresentaram associação módulo-traço.
# Portanto, o WGCNA não atua como filtro estatístico rígido, mas como critério de priorização e organização da rede.

# Identifica quais traits estão disponíveis para uso na seleção, verificando as colunas presentes
available_traits = []

for trait in MODULE_TRAITS_TO_USE:
    has_corr = f"corr_{trait}" in module_ranking_df.columns
    has_pval = f"pval_{trait}" in module_ranking_df.columns
    has_padj = f"padj_{trait}" in module_ranking_df.columns

    if has_corr or has_pval or has_padj:
        available_traits.append(trait)

print("Traits disponíveis:", available_traits)


# Para cada módulo, identifica o trait com melhor evidência (menor padj ou pval) e extrai
# as informações relevantes para a seleção 
def get_best_trait_evidence(row):
    candidates = []

    for trait in available_traits:
        corr = row.get(f"corr_{trait}", np.nan)
        pval = row.get(f"pval_{trait}", np.nan)
        padj = row.get(f"padj_{trait}", np.nan)

        score = padj if pd.notna(padj) else pval

        if pd.notna(score):
            candidates.append({
                "trait": trait,
                "corr": corr,
                "pval": pval,
                "padj": padj,
                "score": score,
            })

    if not candidates:
        return pd.Series({
            "best_module_trait": np.nan,
            "best_module_trait_corr": np.nan,
            "best_module_trait_pval": np.nan,
            "best_module_trait_padj": np.nan,
        })

    best = sorted(candidates, key=lambda item: item["score"])[0]

    return pd.Series({
        "best_module_trait": best["trait"],
        "best_module_trait_corr": best["corr"],
        "best_module_trait_pval": best["pval"],
        "best_module_trait_padj": best["padj"],
    })

# Aplica a função para extrair o melhor trait e suas estatísticas para cada módulo
best_trait_df = module_ranking_df.apply(get_best_trait_evidence, axis=1)

module_ranking_df = pd.concat(
    [module_ranking_df, best_trait_df],
    axis=1,
)

module_ranking_df["trait_support"] = False

# Um módulo ganha suporte de trait se tiver um trait com evidência significativa (padj ou pval abaixo do limiar)
module_ranking_df["trait_support"] = module_ranking_df["trait_support"] | (
    module_ranking_df["best_module_trait_padj"] < MODULE_TRAIT_PADJ_ALPHA
)

module_ranking_df["trait_support"] = module_ranking_df["trait_support"] | (
    module_ranking_df["best_module_trait_pval"] < MODULE_TRAIT_PVAL_ALPHA
)

# Define os módulos selecionados com base nos critérios de número mínimo de DEGs prioritários
# e/ou fração mínima de DEGs, dando suporte à seleção por trait quando disponível
module_ranking_df["module_selected"] = (
    (module_ranking_df["n_deg_priority"] >= MIN_N_DEG_PRIORITY)
    & (
        module_ranking_df["trait_support"]
        | module_ranking_df["fraction_support"]
    )
)

# Para transparência, cria uma coluna que resume quais critérios de seleção cada módulo atende
def build_selection_evidence(row):
    evidence = []

    if row.get("trait_support", False):
        evidence.append("module_trait")

    if row.get("fraction_support", False):
        evidence.append("deg_fraction")

    if not evidence:
        return "not_selected"

    return ";".join(evidence)


module_ranking_df["module_selection_evidence"] = module_ranking_df.apply(
    build_selection_evidence,
    axis=1,
)

# Ordena os módulos para priorizar os que têm suporte mais forte, para facilitar a análise posterior
module_ranking_df = (
    module_ranking_df
    .sort_values(
        [
            "module_selected",
            "trait_support",
            "n_deg_priority",
            "fraction_deg_priority",
        ],
        ascending=[False, False, False, False],
        na_position="last",
    )
    .reset_index(drop=True)
)

selected_modules_df = module_ranking_df.loc[
    module_ranking_df["module_selected"]
].copy()

selected_modules = selected_modules_df["module"].astype(str).tolist()

print("Módulos selecionados:")
print(selected_modules)

display(selected_modules_df)

Traits disponíveis: ['particle_size_um', 'concentration_gL', 'is_100nm']
Módulos selecionados:
['darkgrey', 'dimgrey', 'gainsboro', 'mistyrose', 'indianred', 'white']


,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows,fraction_support,...,padj_is_100nm,significant_is_100nm_padj005,significant_is_100nm_padj010,best_module_trait,best_module_trait_corr,best_module_trait_pval,best_module_trait_padj,trait_support,module_selected,module_selection_evidence
0,darkgrey,3266,203,0.062156,263,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,115,148,True,...,0.059196,False,True,particle_size_um,0.622239,0.002595,0.059196,True,True,module_trait;deg_fraction
1,dimgrey,3883,302,0.077775,443,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,204,239,True,...,0.733196,False,False,particle_size_um,-0.186720,0.417693,0.733196,False,True,deg_fraction
2,gainsboro,1124,46,0.040925,76,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,51,25,True,...,0.593407,False,False,particle_size_um,-0.276454,0.225086,0.593407,False,True,deg_fraction
3,mistyrose,2130,38,0.017840,44,3,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,34,10,True,...,0.517076,False,False,particle_size_um,-0.331082,0.142642,0.517076,False,True,deg_fraction
4,indianred,216,6,0.027778,6,1,MC100_vs_MC1,6,0,True,...,0.733196,False,False,particle_size_um,-0.181986,0.429804,0.733196,False,True,deg_fraction
5,white,317,6,0.018927,7,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,3,4,True,...,0.733196,False,False,particle_size_um,0.198647,0.388015,0.733196,False,True,deg_fraction


## Seleção Final de Genes

In [9]:
# Filtra o dataframe de DEGs prioritários para manter apenas os genes que pertencem aos módulos selecionados
selected_df = priority_deg_df.loc[
    priority_deg_df["module"].astype(str).isin(selected_modules)
].copy()

if selected_df.empty:
    raise ValueError("Nenhum gene selecionado após filtro por módulos.")

selected_df["string_gene_label"] = selected_df["gene_symbol"].fillna(selected_df["gene_id"])

print("Seleção final DEG + WGCNA:")
print("Linhas gene-contraste:", selected_df.shape[0])
print("Genes únicos:", selected_df["gene_id"].nunique())
print("Contrastes representados:", selected_df["contrast"].nunique())
print("Módulos representados:", selected_df["module"].nunique())

display(
    selected_df
    .groupby("module")
    .agg(
        n_rows=("gene_id", "size"),
        n_unique_genes=("gene_id", "nunique"),
        n_contrasts=("contrast", "nunique"),
    )
    .reset_index()
    .sort_values("n_unique_genes", ascending=False)
)

Seleção final DEG + WGCNA:
Linhas gene-contraste: 839
Genes únicos: 601
Contrastes representados: 6
Módulos representados: 6


,module,n_rows,n_unique_genes,n_contrasts
1,dimgrey,443,302,6
0,darkgrey,263,203,5
2,gainsboro,76,46,6
4,mistyrose,44,38,3
3,indianred,6,6,1
5,white,7,6,4


## Construção da Tabela de Nós para o Cytoscape

In [10]:
# Para cada gene, agrega as informações relevantes dos contrastes selecionados
gene_group_cols = [
    "gene_id",
    "gene_symbol",
    "string_gene_label",
]

agg_dict = {
    "primary_module": (
        "module",
        lambda x: x.dropna().astype(str).iloc[0] if x.dropna().shape[0] > 0 else np.nan,
    ),
    "modules": (
        "module",
        lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    ),
    "contrasts": (
        "contrast",
        lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    ),
    "n_selected_rows": ("gene_id", "size"),
    "n_contrasts": ("contrast", "nunique"),
    "min_padj": ("padj", "min"),
    "max_abs_log2FoldChange": ("abs_log2FoldChange", "max"),
    "mean_abs_log2FoldChange": ("abs_log2FoldChange", "mean"),
    "n_up_rows": (
        "regulation",
        lambda x: int((x == "up_in_tested").sum()),
    ),
    "n_down_rows": (
        "regulation",
        lambda x: int((x == "down_in_tested").sum()),
    ),
}

if "gene_name" in selected_df.columns:
    agg_dict["gene_name"] = ("gene_name", "first")

gene_summary_df = (
    selected_df
    .groupby(gene_group_cols, dropna=False)
    .agg(**agg_dict)
    .reset_index()
    .sort_values(
        ["min_padj", "max_abs_log2FoldChange"],
        ascending=[True, False],
    )
)

strongest_effect_df = (
    selected_df
    .sort_values(
        ["gene_id", "abs_log2FoldChange", "padj"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset="gene_id", keep="first")
    [[
        "gene_id",
        "contrast",
        "log2FoldChange",
        "padj",
        "regulation",
    ]]
    .rename(columns={
        "contrast": "strongest_contrast",
        "log2FoldChange": "strongest_log2FoldChange",
        "padj": "strongest_padj",
        "regulation": "strongest_regulation",
    })
)

log2fc_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="log2FoldChange",
        aggfunc="first",
    )
)

log2fc_wide_df.columns = [
    f"log2FC__{col}" for col in log2fc_wide_df.columns
]
log2fc_wide_df = log2fc_wide_df.reset_index()

padj_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="padj",
        aggfunc="first",
    )
)

padj_wide_df.columns = [
    f"padj__{col}" for col in padj_wide_df.columns
]
padj_wide_df = padj_wide_df.reset_index()

regulation_wide_df = (
    selected_df
    .pivot_table(
        index="gene_id",
        columns="contrast",
        values="regulation",
        aggfunc=lambda x: ";".join(
            x.dropna().astype(str).drop_duplicates().sort_values().tolist()
        ),
    )
)

regulation_wide_df.columns = [
    f"regulation__{col}" for col in regulation_wide_df.columns
]
regulation_wide_df = regulation_wide_df.reset_index()

module_support_cols = [
    "module",
    "n_unique_genes_in_module",
    "n_deg_priority",
    "fraction_deg_priority",
    "n_priority_contrasts_present",
    "priority_contrasts_present",
    "fraction_support",
    "trait_support",
    "best_module_trait",
    "best_module_trait_corr",
    "best_module_trait_pval",
    "best_module_trait_padj",
    "module_selection_evidence",
    "module_selected",
]

module_support_cols = [
    col for col in module_support_cols
    if col in module_ranking_df.columns
]

module_support_df = (
    module_ranking_df[module_support_cols]
    .drop_duplicates(subset="module")
    .rename(columns={"module": "primary_module"})
)

cytoscape_node_table_df = (
    gene_summary_df
    .merge(strongest_effect_df, on="gene_id", how="left", validate="one_to_one")
    .merge(log2fc_wide_df, on="gene_id", how="left", validate="one_to_one")
    .merge(padj_wide_df, on="gene_id", how="left", validate="one_to_one")
    .merge(regulation_wide_df, on="gene_id", how="left", validate="one_to_one")
    .merge(module_support_df, on="primary_module", how="left", validate="many_to_one")
)

cytoscape_node_table_df.insert(
    0,
    "node_name",
    cytoscape_node_table_df["string_gene_label"],
)

main_cols = [
    "node_name",
    "gene_id",
    "gene_symbol",
    "string_gene_label",
]

if "gene_name" in cytoscape_node_table_df.columns:
    main_cols.append("gene_name")

main_cols += [
    "primary_module",
    "modules",
    "contrasts",
    "n_selected_rows",
    "n_contrasts",
    "min_padj",
    "max_abs_log2FoldChange",
    "mean_abs_log2FoldChange",
    "strongest_contrast",
    "strongest_log2FoldChange",
    "strongest_padj",
    "strongest_regulation",
    "n_up_rows",
    "n_down_rows",
    "module_selection_evidence",
    "module_selected",
    "best_module_trait",
    "best_module_trait_corr",
    "best_module_trait_pval",
    "best_module_trait_padj",
    "fraction_deg_priority",
]

main_cols = [
    col for col in main_cols
    if col in cytoscape_node_table_df.columns
]

remaining_cols = [
    col for col in cytoscape_node_table_df.columns
    if col not in main_cols
]

cytoscape_node_table_df = cytoscape_node_table_df[
    main_cols + remaining_cols
]

string_gene_list = (
    cytoscape_node_table_df["string_gene_label"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("Tabela de nós Cytoscape:")
print("Linhas:", cytoscape_node_table_df.shape[0])
print("Colunas:", cytoscape_node_table_df.shape[1])
print("Nós únicos:", cytoscape_node_table_df["node_name"].nunique())

print("\nEntradas STRING:", len(string_gene_list))

display(cytoscape_node_table_df.head())

Tabela de nós Cytoscape:
Linhas: 601
Colunas: 50
Nós únicos: 601

Entradas STRING: 601


,node_name,gene_id,gene_symbol,string_gene_label,gene_name,primary_module,modules,contrasts,n_selected_rows,n_contrasts,...,regulation__MA1_vs_CTR,regulation__MC100_vs_MC1,regulation__MC1_vs_CTR,regulation__MD1_vs_CTR,n_unique_genes_in_module,n_deg_priority,n_priority_contrasts_present,priority_contrasts_present,fraction_support,trait_support
0,FAM43A,ENSG00000185112,FAM43A,FAM43A,family with sequence similarity 43 member A,darkgrey,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,...,NaN,down_in_tested,up_in_tested,up_in_tested,3266,203,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,True,True
1,KHSRP,ENSG00000088247,KHSRP,KHSRP,KH-type splicing regulatory protein,dimgrey,dimgrey,MC100_vs_MC1,1,1,...,NaN,down_in_tested,NaN,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
2,SPDL1,ENSG00000040275,SPDL1,SPDL1,spindle apparatus coiled-coil protein 1,dimgrey,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,...,NaN,down_in_tested,up_in_tested,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
3,TOP2A,ENSG00000131747,TOP2A,TOP2A,DNA topoisomerase II alpha,dimgrey,dimgrey,MA100_vs_CTR;MC100_vs_MC1,2,2,...,NaN,down_in_tested,NaN,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
4,CAVIN1,ENSG00000177469,CAVIN1,CAVIN1,caveolae associated protein 1,darkgrey,darkgrey,MC100_vs_MC1,1,1,...,NaN,down_in_tested,NaN,NaN,3266,203,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,True,True


## Salvamento

In [11]:
selected_df.to_csv(DEG_WGCNA_SELECTED_PATH, index=False)
selected_modules_df.to_csv(SELECTED_MODULES_SUMMARY_PATH, index=False)
cytoscape_node_table_df.to_csv(CYTOSCAPE_NODE_TABLE_PATH, index=False)

with open(STRING_INPUT_SELECTED_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(string_gene_list))
    f.write("\n")

print("Arquivos salvos:")
print("-", DEG_WGCNA_SELECTED_PATH)
print("-", SELECTED_MODULES_SUMMARY_PATH)
print("-", STRING_INPUT_SELECTED_PATH)
print("-", CYTOSCAPE_NODE_TABLE_PATH)

Arquivos salvos:
- ../../data/interim/network_export/deg_wgcna_selected_for_string.csv
- ../../data/interim/network_export/selected_modules_summary.csv
- ../../data/interim/network_export/string_input_selected_genes.txt
- ../../data/interim/network_export/cytoscape_node_table.csv


## Resumo

In [12]:
print("Resumo final")
print("============")

print("\nSeleção final DEG + WGCNA")
print("--------------------------")
print(f"Linhas gene-contraste: {selected_df.shape[0]}")
print(f"Genes únicos: {selected_df['gene_id'].nunique()}")
print(f"Contrastes representados: {selected_df['contrast'].nunique()}")
print(f"Módulos representados: {selected_df['module'].nunique()}")
print(f"Entradas STRING: {len(string_gene_list)}")

print("\nMódulos selecionados:")
display(selected_modules_df)

print("\nGenes por módulo:")
display(
    selected_df
    .groupby("module")
    .agg(n_genes=("gene_id", "nunique"))
    .reset_index()
    .sort_values("n_genes", ascending=False)
)

print("\nGenes por contraste:")
display(
    selected_df
    .groupby("contrast")
    .agg(n_genes=("gene_id", "nunique"))
    .reset_index()
    .sort_values("n_genes", ascending=False)
)

print("\nArquivo para STRING:")
print(STRING_INPUT_SELECTED_PATH)

print("\nTabela de nós para Cytoscape:")
print(CYTOSCAPE_NODE_TABLE_PATH)

print("\nResumo da tabela de nós do Cytoscape:")
print("Linhas:", cytoscape_node_table_df.shape[0])
print("Colunas:", cytoscape_node_table_df.shape[1])
print("Nós únicos:", cytoscape_node_table_df["node_name"].nunique())

display(cytoscape_node_table_df.head())

Resumo final

Seleção final DEG + WGCNA
--------------------------
Linhas gene-contraste: 839
Genes únicos: 601
Contrastes representados: 6
Módulos representados: 6
Entradas STRING: 601

Módulos selecionados:


,module,n_unique_genes_in_module,n_deg_priority,fraction_deg_priority,n_priority_gene_contrast_hits,n_priority_contrasts_present,priority_contrasts_present,n_up_priority_rows,n_down_priority_rows,fraction_support,...,padj_is_100nm,significant_is_100nm_padj005,significant_is_100nm_padj010,best_module_trait,best_module_trait_corr,best_module_trait_pval,best_module_trait_padj,trait_support,module_selected,module_selection_evidence
0,darkgrey,3266,203,0.062156,263,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,115,148,True,...,0.059196,False,True,particle_size_um,0.622239,0.002595,0.059196,True,True,module_trait;deg_fraction
1,dimgrey,3883,302,0.077775,443,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,204,239,True,...,0.733196,False,False,particle_size_um,-0.186720,0.417693,0.733196,False,True,deg_fraction
2,gainsboro,1124,46,0.040925,76,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,51,25,True,...,0.593407,False,False,particle_size_um,-0.276454,0.225086,0.593407,False,True,deg_fraction
3,mistyrose,2130,38,0.017840,44,3,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,34,10,True,...,0.517076,False,False,particle_size_um,-0.331082,0.142642,0.517076,False,True,deg_fraction
4,indianred,216,6,0.027778,6,1,MC100_vs_MC1,6,0,True,...,0.733196,False,False,particle_size_um,-0.181986,0.429804,0.733196,False,True,deg_fraction
5,white,317,6,0.018927,7,4,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,3,4,True,...,0.733196,False,False,particle_size_um,0.198647,0.388015,0.733196,False,True,deg_fraction



Genes por módulo:


,module,n_genes
1,dimgrey,302
0,darkgrey,203
2,gainsboro,46
4,mistyrose,38
3,indianred,6
5,white,6



Genes por contraste:


,contrast,n_genes
3,MC100_vs_MC1,415
0,MA100_vs_CTR,168
4,MC1_vs_CTR,148
5,MD1_vs_CTR,93
1,MA100_vs_MA1,12
2,MA1_vs_CTR,3



Arquivo para STRING:
../../data/interim/network_export/string_input_selected_genes.txt

Tabela de nós para Cytoscape:
../../data/interim/network_export/cytoscape_node_table.csv

Resumo da tabela de nós do Cytoscape:
Linhas: 601
Colunas: 50
Nós únicos: 601


,node_name,gene_id,gene_symbol,string_gene_label,gene_name,primary_module,modules,contrasts,n_selected_rows,n_contrasts,...,regulation__MA1_vs_CTR,regulation__MC100_vs_MC1,regulation__MC1_vs_CTR,regulation__MD1_vs_CTR,n_unique_genes_in_module,n_deg_priority,n_priority_contrasts_present,priority_contrasts_present,fraction_support,trait_support
0,FAM43A,ENSG00000185112,FAM43A,FAM43A,family with sequence similarity 43 member A,darkgrey,darkgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR;MD1_vs_CTR,4,4,...,NaN,down_in_tested,up_in_tested,up_in_tested,3266,203,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,True,True
1,KHSRP,ENSG00000088247,KHSRP,KHSRP,KH-type splicing regulatory protein,dimgrey,dimgrey,MC100_vs_MC1,1,1,...,NaN,down_in_tested,NaN,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
2,SPDL1,ENSG00000040275,SPDL1,SPDL1,spindle apparatus coiled-coil protein 1,dimgrey,dimgrey,MA100_vs_CTR;MC100_vs_MC1;MC1_vs_CTR,3,3,...,NaN,down_in_tested,up_in_tested,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
3,TOP2A,ENSG00000131747,TOP2A,TOP2A,DNA topoisomerase II alpha,dimgrey,dimgrey,MA100_vs_CTR;MC100_vs_MC1,2,2,...,NaN,down_in_tested,NaN,NaN,3883,302,6,MA100_vs_CTR;MA100_vs_MA1;MA1_vs_CTR;MC100_vs_...,True,False
4,CAVIN1,ENSG00000177469,CAVIN1,CAVIN1,caveolae associated protein 1,darkgrey,darkgrey,MC100_vs_MC1,1,1,...,NaN,down_in_tested,NaN,NaN,3266,203,5,MA100_vs_CTR;MA100_vs_MA1;MC100_vs_MC1;MC1_vs_...,True,True
